# 92 · Glucose handling along the PT

Kidney collaborators asked whether the continuous PT axis can help explain how SGLT2 inhibitors protect the kidney,
and how glucose handling changes along the nephron. No specimen here was treated with an SGLT2 inhibitor or was
diabetic, so this notebook maps the ground such an experiment would stand on. It shows where along the healthy PT
glucose is taken up, exported and made, in mouse and human, and how ischemic injury changes that map. It then sizes
the drug experiment from these mice. Every gene and set below was named before any profile was looked at.

**Protocol.** `docs/results/pt-glucose-handling.md` (frozen before any fit).

**What bounds the reading.** Two control and two IR male mice, one male human donor (two cortex sections, no outer
medulla). mRNA is not transporter protein or flux. SGLT2 and SGLT1 helped build the coordinate, so statements about
where they change rest on genes that did not (MAP17, GLUT2).

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from IPython.display import display

PKG = results_root / 'package_pseudospace'
EXPORT = {'hvm': PKG / 'hvm' / 'pt_units.h5ad', 'aki': PKG / 'aki' / 'pt_units.h5ad'}
STAGE01 = PKG / 'runs' / 'hvm' / '01_units'
RUN = {'hvm': PKG / 'runs' / 'hvm', 'aki': PKG / 'runs' / 'aki'}
upstream13 = results_root / 'minimal_pt_scfates'
T71 = results_root / 'pt_pathways_package_pseudospace' / 'tables'
T72 = results_root / 'pt_package_coordinate_validation' / 'tables'
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
LCM = data_root / 'external' / 'sglt2i_lcm_vallon'
output = results_root / 'pt_glucose_handling'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)

FROZEN_SHA = {  # sha256 recorded when the protocol was frozen
    T90 / 'a_gene_classes.csv.gz': '48d3394334fd18c7fb7c6b7728bfec49a75303b74be49831b47ffffe6bdf156b',
    T90 / 'd_end_contrasts.csv.gz': 'b4a5646f71f998fa241ef8fb05f30f9a2151a331ebfeb5ed58ac0ea80d9effe1',
    T90 / 'b_set_positions.csv': '907eed01e3b35653ee030bb0ce63593d5c525e8514988e79a574580e0f122f92',
    T90 / 'b_set_positions_fold_protected.csv': 'fd79f411f2ffd50f624f9ca3a90075e597f100cf55c6adaba9c4682c50d352dc',
    T91 / 'i_genes.csv.gz': 'd6ae303b5139ea82c32aefeb6a61276f6d383ea0a562478c8e5159d70540e3b7',
    T91 / 'i5_library_sets.csv': '6502c1984ac2345452d718e779bac16f3966726a97cd1f21af4d9eee7ca4d181',
    T91 / 's2_species_sets.csv': '414bd7d06aa662b8f58243778d622ddde768bbd56fe41aac6d25c05fa30b8636',
    T91 / 'i_induced_genes.csv': 'b3ae25a0393a6cd132ce57ab16ac1b7a03ed26fa587daca3076e7bd499baf6c3',
    T91 / 'i_lost_genes.csv': 'c4468348b0bad9b87f1e1bf45670560522e7cf6b0f89c8eafd991ba5017ae079',
    T71 / 'gene_sets_aki_vs_control.json': '034112512c80d4236497c5f7677829e99c67eafcd33ed645429b16cd93a0cdc4',
    LCM / 'GSE279174' / 'GSE279174_AKITA_S1_kidney_Sema_SGLT2i_TPM_expression_48_samples_removed_oulier.txt.gz':
        'c9c6cafb1f1367eb7f2e8486206b15790089431ab4151ea7718dcd362b87be51',
    LCM / 'GSE279174' / 'GSE279174_series_matrix.txt.gz': 'eaacd8b08cecbb8a4947c118766aaca280f14e13ee7b956d92a2b7c749d1dc34',
    LCM / 'GSE325466' / 'GSE325466_LCM_S3_Kidney_SGLT2i_Sema_log10_TPM_expression_removed_outlier.txt.gz':
        '74f057f80e0b773d1c49842616734ba31e35c4c55785dd98890ee6ba448ae036',
    LCM / 'GSE325466' / 'GSE325466_series_matrix.txt.gz': 'eb8ee419b4e9aebe233e54f694b9afba9edb373d88e460f987e4c62c9d84c72c',
}
REQUIRED = [*EXPORT.values(), *[STAGE01 / f'{s}_units.h5ad' for s in ('Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1')],
            upstream13 / 'ortholog_map_used.csv', T72 / 'transitions.csv', T71 / 'inputs.csv',
            T71 / 'universe_human_vs_mouse.csv', *FROZEN_SHA]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70–72, 90 and 91 first; GEO GSE279174 and '
                                f'GSE325466 go under data/external/sglt2i_lcm_vallon/; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '92.glucose_handling.1'
FINGERPRINT = {'hvm': 'efafc9ac2e99', 'aki': 'bb4a80c1b484'}
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))
SEED = 92
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
AKI4 = CONTROLS + IRS
LABELS = ('PT-S1', 'PT-S2', 'PT-S3')
LABEL_ORIENTATION = ['Cyp2e1', 'Cyp7b1', 'Gatm', 'Slc7a13', 'Slc13a3', 'Slc22a6', 'Slc5a2', 'Slc22a7', 'Slc5a12',
                     'Aqp2', 'Aqp4', 'Atp6v0d2', 'Rhcg', 'Slc14a2', 'Wnt7b']   # 90's; they oriented the coordinate
# Protocol constants
DETECTION, USABLE = .02, .05
N_BOOT, EDGE = 500, .15
N_LIST = (2, 3, 4, 5, 6, 8)
ALPHAS = (.05, .001)
N_STRATA = 10
N_PERM, MIN_SET, Q = 2000, 10, .05
LEVEL, LOCATION, R_REP, RELABEL = .5, .4, .5, .5
LCM_TPM, LCM_LFC, LCM_Q = 5., .5, .05
LCM_P_PH = .01   # post hoc (run record)

# The pre-named genes (protocol section 2): mouse symbol -> (protein, role)
GENES = {
    'Slc5a2': ('SGLT2', 'uptake'), 'Pdzk1ip1': ('MAP17', 'uptake'), 'Slc5a1': ('SGLT1', 'uptake'),
    'Slc2a2': ('GLUT2', 'exit'), 'Slc2a1': ('GLUT1', 'exit'),
    'Slc5a10': ('SGLT5', 'other sugar'), 'Slc5a9': ('SGLT4', 'other sugar'),
    'Slc9a3': ('NHE3', 'sodium'), 'Atp1a1': ('Na/K-ATPase α1', 'sodium'),
    'Pck1': ('PEPCK', 'production'), 'G6pc': ('G6Pase', 'production'), 'Fbp1': ('FBP1', 'production'),
    'Pcx': ('PC', 'production'), 'Aldob': ('ALDOB', 'production'),
    'Hk1': ('HK1', 'glycolysis'), 'Hk2': ('HK2', 'glycolysis'), 'Pfkp': ('PFKP', 'glycolysis'),
    'Pfkfb3': ('PFKFB3', 'glycolysis'), 'Pkm': ('PKM', 'glycolysis'), 'Ldha': ('LDHA', 'glycolysis'),
    'Slc5a12': ('SMCT1', 'lactate'), 'Slc16a1': ('MCT1', 'lactate'), 'Slc16a3': ('MCT4', 'lactate'),
    'Txnip': ('TXNIP', 'glucose sensing'), 'Arrdc4': ('ARRDC4', 'glucose sensing'), 'Mlxipl': ('ChREBP', 'glucose sensing'),
    'Hmgcs2': ('HMGCS2', 'ketone'), 'Bdh1': ('BDH1', 'ketone'), 'Oxct1': ('SCOT', 'ketone'),
    'Insr': ('INSR', 'insulin'),
    'Hnf4a': ('HNF4A', 'regulator'), 'Ppara': ('PPARα', 'regulator'), 'Ppargc1a': ('PGC-1α', 'regulator'),
}
ROLE_ORDER = ['uptake', 'exit', 'other sugar', 'sodium', 'production', 'glycolysis', 'lactate', 'glucose sensing',
              'ketone', 'insulin', 'regulator']
HANDOFF = {'down': ['Slc5a2', 'Pdzk1ip1', 'Slc2a2'], 'up': ['Slc5a1', 'Slc2a1']}
SETS = ['Reactome_2022::Cellular Hexose Transport R-HSA-189200', 'Reactome_2022::Gluconeogenesis R-HSA-70263',
        'KEGG_2019_Mouse::Glycolysis / Gluconeogenesis', 'Reactome_2022::Glycolysis R-HSA-70171',
        'MSigDB_Hallmark_2020::Glycolysis', 'Reactome_2022::Glucose Metabolism R-HSA-70326',
        'MSigDB_Hallmark_2020::mTORC1 Signaling', 'KEGG_2019_Mouse::Insulin signaling pathway',
        'KEGG_2019_Mouse::AMPK signaling pathway',
        'Reactome_2022::Energy Dependent Regulation Of mTOR By LKB1-AMPK R-HSA-380972',
        'Reactome_2022::Pyruvate Metabolism R-HSA-70268', 'KEGG_2019_Mouse::Butanoate metabolism',
        'MSigDB_Hallmark_2020::Hypoxia', 'KEGG_2019_Mouse::HIF-1 signaling pathway', 'KEGG_2019_Mouse::PPAR signaling pathway']

MM = 1 / 25.4
INK, LIGHT, GREY = '#0b0b0b', '#d9d8d4', '#9a9893'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'HUK1_COR1': '#D55E00', 'HUK1_MED1': '#E69F00',
              'IR2A2': '#B2182B', 'IR2A4': '#EF8A62'}
ROLE_COLOR = {'uptake': '#0072B2', 'exit': '#56B4E9', 'other sugar': '#9a9893', 'sodium': '#009E73',
              'production': '#D55E00', 'glycolysis': '#CC79A7', 'glucose sensing': '#E69F00', 'ketone': '#8c510a',
              'lactate': '#b2abd2', 'insulin': '#4d4d4d', 'regulator': '#1b7837'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 200)
GUARDS, READING = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def panel(ax, letter):
    ax.text(-.02, 1.02, letter, transform=ax.transAxes, fontsize=8, fontweight='bold', ha='right', va='bottom')


def seg_lines(ax, cuts=(1, 2), vertical=True):
    for b in cuts:
        (ax.axvline if vertical else ax.axhline)(b, color=LIGHT, lw=.6, zorder=0)


def short(s, n=40):
    name = s.split('::', 1)[-1]
    for tail in (' R-HSA-', ' WP'):
        if tail in name:
            name = name.split(tail)[0]
    return name if len(name) <= n else name[:n - 1] + '…'


def label(g):
    return f'{GENES[g][0]} ({g})' if GENES[g][0].upper() != g.upper() else g


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards (not cached)

G1 the frozen tables; G2 the human-vs-mouse export reproduces 90's inputs (71's universe and 72's transitions); G3
the stage-01 native panels match the export; G4 the AKI export reproduces 91's cuts.

In [ ]:
import anndata as ad
from scipy import sparse
from scipy import stats
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from joblib import Parallel, delayed

from pseudospace import continuous_axis as ca
from pseudospace import glucose_handling as gh
from pseudospace import positional_decomposition as pdc
from pseudospace.positional_decomposition import row_corr
from pseudospace import positional_programs as pp
from pseudospace import species_calls as sc
from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.package_inputs import hvm_inputs, read_pt_units, support_positions
from pseudospace.stage_cache import cached_frame, cached_payload, digest

for path, expected in FROZEN_SHA.items():
    assert sha256(path) == expected, f'G1: {path.name} changed since the protocol was frozen'
GUARDS.append(f'G1: {len(FROZEN_SHA)} frozen inputs match their protocol sha256')

orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
HVM = read_pt_units(EXPORT['hvm'], 'hvm')
assert HVM.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT['hvm']
INP = hvm_inputs(HVM, data_root, orthologs, exclude_genes=MT)
universe71 = pd.read_csv(T71 / 'universe_human_vs_mouse.csv')
assert list(INP['genes']) == universe71[universe71.eligible.astype(bool)].gene.astype(str).tolist(), 'G2: universe differs from 71'
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = np.asarray(INP['segment']).astype(str)
LIBRARY = np.asarray(INP['library'], float)
OBS = INP['obs']
IDENT = OBS['primary_identity'].astype(str).to_numpy()
TRANSITIONS = pd.read_csv(T72 / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, SEGMENT, SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6), 'G2: transitions differ from 72'
MOUSE_CUTS = sc.mean_transitions(TRANSITIONS, MOUSE)
UNITS_MOUSE = sc.to_segment_units(POSITION, MOUSE_CUTS)
GUARDS.append(f'G2: 71\'s universe and 72\'s transitions reproduced; mouse cuts {MOUSE_CUTS[0]:.3f}, {MOUSE_CUTS[1]:.3f}')
HUMAN_CUTS = sc.mean_transitions(TRANSITIONS, HUMAN)
if HUMAN_CUTS is None:
    READING.append('S deviation: the human label transitions are not ordered (HUK1_MED1 places S2→S3 before S1→S2), so '
                   'human segment units do not exist. The human axis is 90\'s 25 equal bins over each section\'s 1–99% '
                   'range, which the protocol named as the check; SP2 compares centres of mass on that rule in both species.')


def equal_bins(position, specimen, names, n_bins=25):
    """90's human rule: per section, n_bins equal bins over its 1–99% range of the coordinate (-1 outside)."""
    b = np.full(len(position), -1)
    for n in names:
        r = specimen == n
        lo, hi = np.quantile(position[r], [.01, .99])
        e = np.linspace(lo, hi, n_bins + 1)
        bb = np.clip(np.searchsorted(e, position[r], side='right') - 1, 0, n_bins - 1)
        bb[(position[r] < lo) | (position[r] > hi)] = -1
        b[r] = bb
    return b, (np.arange(n_bins) + .5) / n_bins

NATIVE, PANEL = {}, {}
for species, names in {'mouse': MOUSE, 'human': HUMAN}.items():
    blocks = []
    for name in names:
        native = ad.read_h5ad(STAGE01 / f'{name}_units.h5ad')
        rows = np.flatnonzero(SPECIMEN == name)
        ids = OBS.polygon_id.astype(str).to_numpy()[rows]
        assert pd.Index(ids).isin(native.obs_names).all() and pd.Index(ids).is_unique, f'G3: {name} units missing'
        x = sparse.csr_matrix(native[list(ids)].X, dtype=float)
        export_x = sparse.csr_matrix(HVM[OBS.index[rows]].X, dtype=float)
        symbols = HVM.var_names if species == 'mouse' else pd.Index(HVM.var.human_symbol.astype(str))
        cols = native.var_names.get_indexer(symbols)
        assert (cols >= 0).all() and (x[:, cols] != export_x).nnz == 0, f'G3: {name} counts differ from the export'
        PANEL[species] = pd.Index(native.var_names.astype(str))
        blocks.append((rows, x))
    NATIVE[species] = {'rows': np.concatenate([r for r, _ in blocks]),
                       'counts': sparse.vstack([x for _, x in blocks]).tocsr()}
GUARDS.append('G3: every export unit is in its stage-01 file; shared genes identical')

BUILT = {}
for run in ('hvm', 'aki'):
    hv = set()
    for name in ('pass1_labelled.h5ad', 'pass2.h5ad'):
        obj = ad.read_h5ad(RUN[run] / '03_integration' / name, backed='r')
        hv |= set(obj.var_names[obj.var['highly_variable_for_harmony'].to_numpy(bool)].astype(str))
        obj.file.close()
    BUILT[run] = hv | set(LABEL_ORIENTATION)
TO_HUMAN = pd.Series(HVM.var.human_symbol.astype(str).to_numpy(), index=HVM.var_names.astype(str))
HUMAN_SYMBOL = {g: TO_HUMAN.get(g, {'G6pc': 'G6PC1'}.get(g, g.upper())) for g in GENES}
print('\n'.join('· ' + g for g in GUARDS))

## 3 · Healthy profiles (90's construction)

Mouse: PT units of the two controls, fixed 0.1 bins of mouse segment units on [0, 2.5); per mouse each gene's log2
rate per bin minus its mean, minus the reference (the median usable gene per bin). Human: the same in human segment
units (the human sections' own label transitions), and 90's 25 equal bins as a check.

In [ ]:
U_PT = IDENT == 'Proximal_Tubule'


def species_rows(species):
    rows = NATIVE[species]['rows']
    keep = U_PT[rows]
    return rows[keep], NATIVE[species]['counts'][keep]


H = {}
for species, names in {'mouse': MOUSE, 'human': HUMAN}.items():
    rows, x = species_rows(species)
    spec = SPECIMEN[rows]
    det = np.array([np.asarray((x[spec == n] > 0).mean(axis=0)).ravel() for n in names])
    not_mt = ~PANEL[species].str.upper().str.startswith('MT-')
    expressed = (det >= DETECTION).all(axis=0) & not_mt
    genes = np.flatnonzero(expressed)
    H[species] = {'rows': rows, 'counts': x[:, genes], 'genes': list(PANEL[species][genes]), 'specimen': spec,
                  'library': LIBRARY[rows], 'segment': SEGMENT[rows],
                  'position': POSITION[rows], 'usable': (det[:, genes] >= USABLE).all(axis=0),
                  'detection': pd.DataFrame(det.T, index=PANEL[species], columns=names)}
    H[species]['index'] = pd.Series(np.arange(len(genes)), index=H[species]['genes'])
    if species == 'mouse':
        b, centres = pp.fixed_bins(UNITS_MOUSE[rows])
    else:
        b, centres = equal_bins(POSITION[rows], spec, names)
    H[species]['bin'], H[species]['centres'] = b, centres
    H[species]['profiles'] = pp.bin_profiles(H[species]['counts'], H[species]['library'], b, len(centres),
                                             spec, names)
    H[species]['reference'] = {n: pp.reference_profile(H[species]['profiles'][n], H[species]['usable']) for n in names}
    H[species]['deviation'] = {n: H[species]['profiles'][n] - H[species]['reference'][n] for n in names}
    H[species]['weight'] = {n: np.bincount(b[(spec == n) & (b >= 0)], minlength=len(centres)).astype(float) for n in names}
    H[species]['frac'] = {n: ca.label_fractions(b[spec == n], H[species]['segment'][spec == n], len(centres), LABELS)
                          for n in names}
CENTRES = H['mouse']['centres']
NB = len(CENTRES)
m, hh = H['mouse'], H['human']
H_CENTRES = hh['centres']
assert np.all([np.isfinite(m['reference'][n]) for n in MOUSE]), 'bins without enough units in a mouse'
# SP2: the mouse on the human rule (equal bins of the shared coordinate), for a like-for-like centre of mass
m_eb, _ = equal_bins(m['position'], m['specimen'], MOUSE)
m_eq_prof = pp.bin_profiles(m['counts'], m['library'], m_eb, len(H_CENTRES), m['specimen'], MOUSE)
M_EQ_DEV = {n: m_eq_prof[n] - pp.reference_profile(m_eq_prof[n], m['usable']) for n in MOUSE}

A90 = pd.read_csv(T90 / 'a_gene_classes.csv.gz')
assert A90.gene.astype(str).tolist() == list(np.array(m['genes'])[m['usable']]), 'G5: usable mouse genes differ from 90'
GUARDS.append(f"G5: {m['usable'].sum():,} usable mouse genes equal 90's ladder universe")

DETECT = pd.DataFrame({
    'protein': [GENES[g][0] for g in GENES], 'role': [GENES[g][1] for g in GENES],
    'human symbol': [HUMAN_SYMBOL[g] for g in GENES],
    **{f'det {n}': [m['detection'].at[g, n] if g in m['detection'].index else np.nan for g in GENES] for n in MOUSE},
    **{f'det {n}': [hh['detection'].at[HUMAN_SYMBOL[g], n] if HUMAN_SYMBOL[g] in hh['detection'].index else np.nan
                    for g in GENES] for n in HUMAN},
    'built (human vs mouse)': [g in BUILT['hvm'] for g in GENES], 'built (AKI)': [g in BUILT['aki'] for g in GENES]},
    index=list(GENES))
DETECT['mouse usable'] = [g in m['index'].index and bool(m['usable'][m['index'][g]]) for g in GENES]
DETECT['human usable'] = [HUMAN_SYMBOL[g] in hh['index'].index and bool(hh['usable'][hh['index'][HUMAN_SYMBOL[g]]])
                          for g in GENES]
display(DETECT.round(3))
print(f"Healthy mouse PT units {pd.Series(m['specimen']).value_counts().to_dict()}; human "
      f"{pd.Series(hh['specimen']).value_counts().to_dict()}")

code_key = digest([NOTEBOOK_LOGIC_VERSION, Path(gh.__file__).read_text(), Path(ca.__file__).read_text(),
                   Path(pp.__file__).read_text()])
SECONDS = {}


def cached_table(stage, compute, *, params=None, inputs=None):
    t = time.perf_counter()
    out = cached_frame(stage, compute, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = time.perf_counter() - t
    return out


def cached(stage, compute, *, params=None, inputs=None):
    t = time.perf_counter()
    out = cached_payload(stage, compute, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = time.perf_counter() - t
    return out

## 4 · H1 · The healthy mouse map: each gene's shape (90's ladder, read, not refitted)

In [ ]:
H1 = A90.set_index('gene').reindex([g for g in GENES if DETECT.at[g, 'mouse usable']])
H1 = H1[['class', 'continuous vs discrete', 'dBIC continuous', 'location', 'switch width', 'S position', 'S width',
         'S amp', 'B position', 'L slope', 'D S2-S1', 'D S3-S1', 'fold-protected', 'log2 CP10k', 'class Ctrl1A2',
         'class Ctrl1A4']]
H1.insert(0, 'protein', [GENES[g][0] for g in H1.index])
H1.insert(1, 'role', [GENES[g][1] for g in H1.index])
H1['centre of mass'] = [float(np.mean([ca.centre_of_mass(m['deviation'][n][m['index'][g]], CENTRES)[0] for n in MOUSE]))
                        for g in H1.index]
display(H1.round(2))
H1.to_csv(tables / 'h1_gene_shapes.csv')

## 5 · H2 · Where uptake hands over: switch midpoints with a unit bootstrap

Per control mouse: the switch model of 90's ladder (a logistic with midpoint and 10–90% width) on each handoff gene's
deviation profile, and on 500 resamplings of the mouse's units. The handoff is read from fold-protected genes
(MAP17, GLUT2) against SGLT1; SGLT2 and GLUT1 helped build the coordinate. The same on the AKI export's controls.

In [ ]:
H2_GENES = [g for g in HANDOFF['down'] + HANDOFF['up'] if DETECT.at[g, 'mouse usable']]


def switch_table(dev, weight, frac):
    """90's ladder (single mouse) on rows of ``dev``; returns S position, S width, S amp and the class. Guard: the
    bootstrap's switch-only fit gives the same switch."""
    t = ca.fit_ladder([dev], [weight], CENTRES, [frac])
    c = ca.ladder_class(t)
    pos, wid, _ = gh.switch_fit(dev, weight, CENTRES)
    assert np.allclose(pos, t['S position']) and np.allclose(wid, t['S width']), 'G8: switch fits disagree'
    return t[['S position', 'S width', 'S amp']].assign(**{'class': c['class'].to_numpy()})


def boot_compute(H_, names, tag):
    def run():
        out = {}
        for k, n in enumerate(names):
            r = H_['specimen'] == n
            targets = [H_['index'][g] for g in H2_GENES]
            boot = gh.bootstrap_deviation(H_['counts'][r], H_['library'][r], H_['bin'][r], NB, targets, H_['usable'],
                                          n_boot=N_BOOT, seed=SEED + k)
            fits = np.array([gh.switch_fit(draw, H_['weight'][n], CENTRES) for draw in boot])   # draws x 3 x genes
            out[f'{n} position'], out[f'{n} width'], out[f'{n} amp'] = fits[:, 0], fits[:, 1], fits[:, 2]
        return out
    return run


H2_ROWS = []
H2_BOOT = {}
for n in MOUSE:
    d = m['deviation'][n][[m['index'][g] for g in H2_GENES]]
    t = switch_table(d, m['weight'][n], m['frac'][n])
    for g, (_, row) in zip(H2_GENES, t.iterrows()):
        H2_ROWS.append({'export': 'human vs mouse', 'mouse': n, 'gene': g, 'protein': GENES[g][0],
                        'direction': 'down' if g in HANDOFF['down'] else 'up',
                        'fold-protected': g not in BUILT['hvm'], **row.to_dict()})
H2_BOOT['hvm'] = cached('h2_boot_hvm', boot_compute(m, MOUSE, 'hvm'), params={'n_boot': N_BOOT, 'seed': SEED},
                        inputs={'genes': H2_GENES, 'bins': m['bin'], 'lib': m['library']})
H2 = pd.DataFrame(H2_ROWS)


def add_intervals(table, boot, export):
    for i, row in table[table.export.eq(export)].iterrows():
        j = H2_GENES.index(row.gene)
        p = boot[f'{row.mouse} position'][:, j]
        table.loc[i, 'position 2.5%'], table.loc[i, 'position 97.5%'] = np.nanquantile(p, [.025, .975])
        w = boot[f'{row.mouse} width'][:, j]
        table.loc[i, 'width 2.5%'], table.loc[i, 'width 97.5%'] = np.nanquantile(w, [.025, .975])
    return table


def place(p):
    near = [k for k in (1, 2) if abs(p - k) <= EDGE]
    return f'at S{near[0]}/S{near[0] + 1}' if near else f'inside S{int(np.clip(np.floor(p), 0, 2)) + 1}'


H2 = add_intervals(H2, H2_BOOT['hvm'], 'human vs mouse')
H2['place'] = H2['S position'].map(place)
display(H2.round(2))

## 6 · S · Human: the same genes along the human PT

In [ ]:
S_ROWS = []
D90 = pd.read_csv(T90 / 'd_end_contrasts.csv.gz')
D90 = D90.set_index('mouse')
for g in GENES:
    hs = HUMAN_SYMBOL[g]
    row = {'gene': g, 'protein': GENES[g][0], 'human symbol': hs, 'human usable': DETECT.at[g, 'human usable']}
    if g in D90.index:
        r = D90.loc[g]
        row.update({'E mouse': r['E mouse'], 'E human': r['E human'], 'category': r['category'],
                    'strength': r['strength']})
    if row['human usable']:
        j = hh['index'][hs]
        row['centre of mass (human, equal bins)'] = float(np.mean([ca.centre_of_mass(hh['deviation'][n][j], H_CENTRES)[0]
                                                                  for n in HUMAN]))
    if g in m['index'].index:
        row['centre of mass (mouse, equal bins)'] = float(np.mean([ca.centre_of_mass(M_EQ_DEV[n][m['index'][g]], H_CENTRES)[0]
                                                                  for n in MOUSE]))
        row['r COR1×MED1'] = float(np.corrcoef(*[np.where(np.isfinite(hh['deviation'][n][j]), hh['deviation'][n][j], 0.)
                                                 for n in HUMAN])[0, 1])
    S_ROWS.append(row)
S = pd.DataFrame(S_ROWS).set_index('gene')
display(S.round(2))
S.to_csv(tables / 's_species.csv')

## 7 · I · Injury: 91's decomposition for the pre-named genes and sets (read, not refitted)

In [ ]:
I91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
INDUCED91 = pd.read_csv(T91 / 'i_induced_genes.csv')
LOST91 = pd.read_csv(T91 / 'i_lost_genes.csv')
cols = ['L', 'M', 'r_P', 'L R1', 'L R2', 'M R1', 'M R2', 'call', 'peak', 'Δ late', 'injury class', 'injury location',
        'zonated', 'A condition', 'A R1', 'A R2', 'survival']
I_G = I91.reindex(list(GENES))[cols]
I_G.insert(0, 'protein', [GENES[g][0] for g in GENES])
I_G.insert(1, 'role', [GENES[g][1] for g in GENES])
I_G['induced'] = I_G.index.isin(INDUCED91.gene.astype(str))
I_G['lost'] = I_G.index.isin(LOST91.gene.astype(str))
display(I_G.round(2))
I_G.to_csv(tables / 'i_genes.csv')

LIB91 = pd.read_csv(T91 / 'i5_library_sets.csv')
LIB91['rank |z L|'] = LIB91['z L'].abs().rank(ascending=False).astype(int)
LIB91['rank z M'] = LIB91['z M'].rank(ascending=False).astype(int)
I_S = LIB91.set_index('set').reindex(SETS)
I_S = I_S[['members', 'L', 'z L', 'q L', 'rank |z L|', 'M', 'z M', 'q M', 'rank z M', 'level set', 'location set',
           'peak', 'A', 'z A', 'gradient']]
I_S.insert(0, 'name', [short(s) for s in SETS])
print(f'{len(LIB91)} library sets in 91')
display(I_S.round(3))
I_S.to_csv(tables / 'i_sets.csv')
B90 = pd.read_csv(T90 / 'b_set_positions.csv').set_index('set').reindex(SETS)
B90_FP = pd.read_csv(T90 / 'b_set_positions_fold_protected.csv').set_index('set').reindex(SETS)
H_S = pd.DataFrame({'name': [short(s) for s in SETS], 'CoM Ctrl1A2': B90['CoM Ctrl1A2'], 'CoM Ctrl1A4': B90['CoM Ctrl1A4'],
                    'q': B90['q'], 'direction': B90['direction'], 'positioned': B90['positioned'],
                    'positioned (fold-protected)': B90_FP['positioned'], 'direction (fold-protected)': B90_FP['direction']})
SP91 = pd.read_csv(T91 / 's2_species_sets.csv').set_index('set').reindex(SETS)
H_S['species C'], H_S['species z C'], H_S['species q C'] = SP91['C'], SP91['z C'], SP91['q C']
display(H_S.round(3))
H_S.to_csv(tables / 'h_sets.csv')

### The AKI export: Δ(x) for the figure, the handoff on its controls, and the design numbers

In [ ]:
AKI = read_pt_units(EXPORT['aki'], 'aki')
assert AKI.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT['aki']
aobs = AKI.obs
a_spec_all = aobs['sample'].astype(str).to_numpy()
a_seg_all = aobs.segment_class.astype(str).to_numpy()
A_MASK, a_pos = support_positions(aobs.pseudospace.to_numpy(float), a_spec_all, a_seg_all)
A_SPEC, A_SEG = a_spec_all[A_MASK], a_seg_all[A_MASK]
A_IDENT = aobs.primary_identity.astype(str).to_numpy()[A_MASK]
A_TR = logistic_transitions(a_pos, A_SEG, A_SPEC)
A_CUTS = sc.mean_transitions(A_TR, CONTROLS)
assert np.allclose(np.round(A_CUTS, 3), (.339, .657)), f'G4: cuts {A_CUTS} differ from 91'
A_UNITS = sc.to_segment_units(a_pos, A_CUTS)
GUARDS.append(f'G4: AKI control cuts {A_CUTS[0]:.3f}, {A_CUTS[1]:.3f} equal 91\'s')
A_X = sparse.csr_matrix(AKI.X, dtype=float)[np.flatnonzero(A_MASK)]
A_GENES_ALL = pd.Index(AKI.var_names.astype(str))
A_NOT_MT = ~A_GENES_ALL.str.startswith('mt-')
A_LIB = np.asarray(A_X[:, np.flatnonzero(A_NOT_MT)].sum(axis=1)).ravel()
A_PT = A_IDENT == 'Proximal_Tubule'
ax_, aspec, aunits, aseg, alib = A_X[np.flatnonzero(A_PT)], A_SPEC[A_PT], A_UNITS[A_PT], A_SEG[A_PT], A_LIB[A_PT]
A_DET = np.array([np.asarray((ax_[aspec == n] > 0).mean(axis=0)).ravel() for n in AKI4])
A_SHARED = (A_DET >= DETECTION).all(axis=0) & A_NOT_MT
A_SH = np.flatnonzero(A_SHARED)
A_GENES = np.array(A_GENES_ALL[A_SH])
assert A_GENES.tolist() == I91.index.astype(str).tolist(), 'G6: shared AKI genes differ from 91'
A_REF = (A_DET[:, A_SH] >= USABLE).all(axis=0)
A_BIN, A_CENTRES = pp.fixed_bins(aunits)
assert np.allclose(A_CENTRES, CENTRES)
A_RATES = pp.bin_profiles(ax_[:, A_SH], alib, A_BIN, NB, aspec, AKI4, centre=False)
A_VALID = np.all([np.isfinite(A_RATES[n][0]) for n in AKI4], axis=0)
for n in AKI4:
    A_RATES[n][:, ~A_VALID] = np.nan
A_INDEX = pd.Series(np.arange(len(A_GENES)), index=A_GENES)
DELTA = pp.centre_by_median(np.mean([A_RATES[n] for n in IRS], axis=0) - np.mean([A_RATES[n] for n in CONTROLS], axis=0), A_REF)
L_CHECK = np.nanmean(DELTA, axis=1)
assert np.allclose(L_CHECK, I91['L'].to_numpy(), atol=1e-9), 'G6: Δ does not reproduce 91\'s L'
GUARDS.append('G6: the AKI export reproduces 91\'s shared genes and level L')

# H2 on the AKI export's controls (profiles centred and referenced as in section 3)
AH = {'counts': ax_[:, A_SH], 'library': alib, 'bin': A_BIN, 'specimen': aspec, 'usable': A_REF, 'index': A_INDEX,
      'segment': aseg}
AH['profiles'] = pp.bin_profiles(AH['counts'], alib, A_BIN, NB, aspec, AKI4)
AH['deviation'] = {n: AH['profiles'][n] - pp.reference_profile(AH['profiles'][n], A_REF) for n in AKI4}
AH['weight'] = {n: np.bincount(A_BIN[(aspec == n) & (A_BIN >= 0)], minlength=NB).astype(float) for n in AKI4}
AH['frac'] = {n: ca.label_fractions(A_BIN[aspec == n], aseg[aspec == n], NB, LABELS) for n in AKI4}
for n in CONTROLS:
    d = AH['deviation'][n][[A_INDEX[g] for g in H2_GENES]]
    t = switch_table(d, AH['weight'][n], AH['frac'][n])
    for g, (_, row) in zip(H2_GENES, t.iterrows()):
        H2_ROWS.append({'export': 'AKI', 'mouse': n, 'gene': g, 'protein': GENES[g][0],
                        'direction': 'down' if g in HANDOFF['down'] else 'up',
                        'fold-protected': g not in BUILT['aki'], **row.to_dict()})
H2_BOOT['aki'] = cached('h2_boot_aki', boot_compute(AH, CONTROLS, 'aki'), params={'n_boot': N_BOOT, 'seed': SEED},
                        inputs={'genes': H2_GENES, 'bins': A_BIN, 'lib': alib})
H2 = add_intervals(pd.DataFrame(H2_ROWS), H2_BOOT['hvm'], 'human vs mouse')
H2 = add_intervals(H2, H2_BOOT['aki'], 'AKI')
H2['place'] = H2['S position'].map(place)
display(H2.round(2))
H2.to_csv(tables / 'h2_switches.csv', index=False)


def gap(table, export, n, a, b):
    t = table[table.export.eq(export) & table.mouse.eq(n)].set_index('gene')
    return float(t.at[b, 'S position'] - t.at[a, 'S position'])


H2_GAPS = pd.DataFrame([{'export': e, 'mouse': n, 'pair': f'{GENES[a][0]} → {GENES[b][0]}',
                         'gap (units)': gap(H2, e, n, a, b)}
                        for e, names in (('human vs mouse', MOUSE), ('AKI', CONTROLS)) for n in names
                        for a, b in (('Pdzk1ip1', 'Slc5a1'), ('Slc2a2', 'Slc5a1'), ('Slc5a2', 'Slc5a1'), ('Slc2a2', 'Slc2a1'))
                        if a in H2_GENES and b in H2_GENES])
display(H2_GAPS.round(2))
H2_GAPS.to_csv(tables / 'h2_gaps.csv', index=False)

# Post hoc label-only check: detection by H&E segment label, which does not use the coordinate
H2_LABEL = pd.DataFrame({(n, lab): [float((m['counts'][(m['specimen'] == n) & (m['segment'] == lab)][:, m['index'][g]] > 0).mean())
                                    for g in H2_GENES] for n in MOUSE for lab in LABELS}, index=[GENES[g][0] for g in H2_GENES])
READING.append('H2 post hoc: detection of the handoff genes in units of each H&E segment label (no coordinate), because '
               'SGLT2 and SGLT1 helped build the coordinate.')
display(H2_LABEL.round(3))
H2_LABEL.to_csv(tables / 'h2_detection_by_label.csv')

## 8 · D · Sizing an SGLT2-inhibitor experiment from these mice

Per mouse, each shared gene's level ℓ (mean over valid bins of its binned log2 rate, centred per bin by the median
reference gene) and location profile (the centred rate minus ℓ). The within-condition SD comes from the two controls
and the two IR mice (2 df per gene), pooled within expression deciles by its median. The minimum detectable effect
is that of a two-sided two-sample t-test with 80% power.

In [ ]:
CENTRED = {n: A_RATES[n] - np.nanmedian(A_RATES[n][A_REF], axis=0) for n in AKI4}
LEVEL_N = np.column_stack([np.nanmean(CENTRED[n], axis=1) for n in AKI4])
LOC_N = {n: CENTRED[n] - LEVEL_N[:, [k]] for k, n in enumerate(AKI4)}
A_LOGCP = np.log2(np.asarray(ax_[np.isin(aspec, CONTROLS)][:, A_SH].sum(axis=0)).ravel()
                  / alib[np.isin(aspec, CONTROLS)].sum() * 1e4 + 1e-6)
STRATA = pd.qcut(A_LOGCP, N_STRATA, labels=False, duplicates='drop')
VAR_L, DF_L, SD_L = gh.within_group_sd(LEVEL_N, [[0, 1], [2, 3]], STRATA)
LOC_VAR = np.nanmean([np.nanmean((LOC_N[a] - LOC_N[b]) ** 2, axis=1) / 2 for a, b in (CONTROLS, IRS)], axis=0)
SD_LOC = {s: float(np.sqrt(np.nanmedian(LOC_VAR[STRATA == s]))) for s in np.unique(STRATA)}

D1 = pd.DataFrame([{'mice per group': n, 'other splits': gh.balanced_relabelings(n)[0],
                    'smallest permutation p': gh.balanced_relabelings(n)[1]} for n in N_LIST])
display(D1.round(4))
D_STRATA = pd.DataFrame({'stratum': list(SD_L), 'median log2 CP10k': [float(np.median(A_LOGCP[STRATA == s])) for s in SD_L],
                         'SD level': list(SD_L.values()), 'SD location (RMS)': [SD_LOC[s] for s in SD_L]})
for a in ALPHAS:
    for n in N_LIST[1:]:
        D_STRATA[f'MDE level n={n} α={a}'] = [gh.minimum_detectable_effect(sd, n, alpha=a) for sd in D_STRATA['SD level']]
display(D_STRATA.round(3))
D_GENES = pd.DataFrame({'protein': [GENES[g][0] for g in GENES if g in A_INDEX.index],
                        'stratum': [int(STRATA[A_INDEX[g]]) for g in GENES if g in A_INDEX.index],
                        'gene SD level (2 df)': [float(np.sqrt(VAR_L[A_INDEX[g]])) for g in GENES if g in A_INDEX.index],
                        'gene SD location (RMS)': [float(np.sqrt(LOC_VAR[A_INDEX[g]])) for g in GENES if g in A_INDEX.index]},
                       index=[g for g in GENES if g in A_INDEX.index])
D_GENES['stratum SD level'] = [SD_L[s] for s in D_GENES.stratum]
for n in (3, 4, 6):
    D_GENES[f'MDE level n={n}'] = [gh.minimum_detectable_effect(sd, n) for sd in D_GENES['stratum SD level']]
display(D_GENES.round(3))
D_STRATA.to_csv(tables / 'd_strata.csv', index=False)
D_GENES.to_csv(tables / 'd_genes.csv')
D1.to_csv(tables / 'd_relabelings.csv', index=False)

## 9 · X · The drug's own footprint: microdissected S1 and S3 of dapagliflozin-treated mice

GSE279174 (SGLT2-positive early PT) and GSE325466 (LTL-positive, SGLT2-negative late PT in deep cortex and outer
medulla), laser-captured from the same male DBA mice: wild type or Akita, vehicle or dapagliflozin for 2 weeks
(Kim/Vallon 2025, 2026). X0 checks our axis against the two pure positions. X1–X3 map the drug's programs onto the
healthy axis and the injury decomposition.

In [ ]:
import gzip
import csv


def lcm_samples(path):
    rows = {}
    with gzip.open(path, 'rt') as f:
        for line in f:
            if line.startswith('!Sample_title') or line.startswith('!Sample_characteristics_ch1'):
                parts = next(csv.reader([line.rstrip('\n')], delimiter='\t'))
                if parts[0] == '!Sample_title':
                    rows['title'] = [v.replace('S3-', '') for v in parts[1:]]
                else:
                    rows[parts[1].split(':')[0]] = [v.split(': ', 1)[1] for v in parts[1:]]
    t = pd.DataFrame(rows).set_index('title')
    t['diabetic'] = t.genotype.str.contains('Akita') if 'disease state' not in t else t['disease state'].eq('diabetic')
    t['sglt1'] = np.where(t.genotype.str.startswith('Sglt1 KO'), 'KO', 'WT')
    t['group'] = (pd.Series(np.where(t.diabetic, 'Akita', 'WT'), index=t.index) + ' '
                  + t.treatment.map({'Vehicle': 'veh', 'Dapagliflozin': 'dapa', 'Semaglutide': 'sema'}))
    return t


def lcm_tpm(path, log10):
    t = pd.read_csv(path, sep='\t', index_col=0)
    if log10:
        t = np.power(10., t).fillna(0.)
        t.index = t.index.str.rsplit('_', n=1).str[0]
    t = t.groupby(level=0).sum()
    return t / t.sum() * 1e6


LCM_META = {'S1': lcm_samples(LCM / 'GSE279174' / 'GSE279174_series_matrix.txt.gz'),
            'S3': lcm_samples(LCM / 'GSE325466' / 'GSE325466_series_matrix.txt.gz')}
LCM_TPM_ = {'S1': lcm_tpm(LCM / 'GSE279174' / 'GSE279174_AKITA_S1_kidney_Sema_SGLT2i_TPM_expression_48_samples_removed_oulier.txt.gz', False),
            'S3': lcm_tpm(LCM / 'GSE325466' / 'GSE325466_LCM_S3_Kidney_SGLT2i_Sema_log10_TPM_expression_removed_outlier.txt.gz', True)}
for seg in ('S1', 'S3'):
    assert set(LCM_TPM_[seg].columns) <= set(LCM_META[seg].index), f'G7: {seg} columns without metadata'
    LCM_META[seg] = LCM_META[seg].loc[LCM_TPM_[seg].columns]
GUARDS.append('G7: LCM samples ' + json.dumps({seg: LCM_META[seg][LCM_META[seg].sglt1.eq('WT')].group.value_counts().to_dict()
                                               for seg in LCM_META}))

# X0: the S3 − S1 contrast in the wild-type vehicle mice present in both tables, on genes of both tables
common = LCM_TPM_['S1'].index.intersection(LCM_TPM_['S3'].index)
veh = [c for c in LCM_META['S1'].index[LCM_META['S1'].group.eq('WT veh') & LCM_META['S1'].sglt1.eq('WT')]
       if c in LCM_TPM_['S3'].columns]
both = {seg: LCM_TPM_[seg].loc[common, veh] for seg in ('S1', 'S3')}
both = {seg: v / v.sum() * 1e6 for seg, v in both.items()}
expressed = pd.concat(both.values(), axis=1).mean(axis=1) >= LCM_TPM
LCM_D = (np.log2(both['S3'] + 1) - np.log2(both['S1'] + 1)).mean(axis=1)[expressed]
x0 = pd.DataFrame({'LCM S3 − S1': LCM_D}).join(D90[['E mouse', 'fold-protected']], how='inner')
x0 = x0.join(A90.set_index('gene')[['D S3-S1']], how='left')
rng0 = np.random.default_rng(SEED)
X0 = []
for name, rows in (('all genes', x0.index), ('fold-protected', x0.index[x0['fold-protected'].astype(bool)])):
    sub = x0.loc[rows].dropna(subset=['E mouse'])
    rho = spearmanr(sub['LCM S3 − S1'], sub['E mouse']).statistic
    boot = [spearmanr(*sub.iloc[rng0.integers(0, len(sub), len(sub))][['LCM S3 − S1', 'E mouse']].to_numpy().T).statistic
            for _ in range(1000)]
    sd = sub.dropna(subset=['D S3-S1'])
    X0.append({'genes': name, 'n': len(sub), 'rho with end contrast E': rho,
               '2.5%': np.quantile(boot, .025), '97.5%': np.quantile(boot, .975),
               'rho with label step D S3-S1': spearmanr(sd['LCM S3 − S1'], sd['D S3-S1']).statistic, 'n label': len(sd)})
X0 = pd.DataFrame(X0)
print(f'X0: {len(veh)} wild-type vehicle mice in both tables; {int(expressed.sum()):,} expressed common genes')
display(X0.round(3))
X0.to_csv(tables / 'x0_lcm_validation.csv', index=False)

# The drug's programs: within each table, wild-type Sglt1 mice only
CONTRASTS_X = {'dapagliflozin (non-diabetic)': ('WT dapa', 'WT veh'), 'diabetes': ('Akita veh', 'WT veh'),
               'dapagliflozin (diabetic)': ('Akita dapa', 'Akita veh')}
X_DE, X_PROG, X_PROG_PH = {}, {}, {}
for seg in ('S1', 'S3'):
    meta, tpm = LCM_META[seg], LCM_TPM_[seg]
    for cname, (case, ref) in CONTRASTS_X.items():
        a = tpm.loc[:, meta.index[meta.group.eq(case) & meta.sglt1.eq('WT')]]
        b = tpm.loc[:, meta.index[meta.group.eq(ref) & meta.sglt1.eq('WT')]]
        keep = pd.concat([a, b], axis=1).mean(axis=1) >= LCM_TPM
        la, lb = np.log2(a[keep] + 1), np.log2(b[keep] + 1)
        de = pd.DataFrame({'lfc': la.mean(axis=1) - lb.mean(axis=1),
                           'p': stats.ttest_ind(la, lb, axis=1, equal_var=False).pvalue}, index=la.index)
        de['q'] = multipletests(de.p.fillna(1.), method='fdr_bh')[1]
        X_DE[(seg, cname)] = de.assign(n_case=a.shape[1], n_ref=b.shape[1])
        X_PROG[f'{cname} {seg}: up'] = de.index[(de.q <= LCM_Q) & (de.lfc >= LCM_LFC)].tolist()
        X_PROG[f'{cname} {seg}: down'] = de.index[(de.q <= LCM_Q) & (de.lfc <= -LCM_LFC)].tolist()
        X_PROG_PH[f'{cname} {seg}: up'] = de.index[(de.p <= LCM_P_PH) & (de.lfc >= LCM_LFC)].tolist()
        X_PROG_PH[f'{cname} {seg}: down'] = de.index[(de.p <= LCM_P_PH) & (de.lfc <= -LCM_LFC)].tolist()
X_DIAG = pd.DataFrame([{'segment': seg, 'contrast': c, 'mice': f"{int(de.n_case.iloc[0])} vs {int(de.n_ref.iloc[0])}",
                        'genes': len(de), 'p ≤ 0.01': int((de.p <= .01).sum()), 'expected by chance': round(.01 * len(de)),
                        'min q': de.q.min()} for (seg, c), de in X_DE.items()])
display(X_DIAG.round(3))
X_DIAG.to_csv(tables / 'x_lcm_de_summary.csv', index=False)
print('frozen rule:', {k: len(v) for k, v in X_PROG.items()})
print('post hoc rule:', {k: len(v) for k, v in X_PROG_PH.items()})
READING.append('X post hoc: the frozen program rule (BH q ≤ 0.05) leaves every program but one empty (Welch t on TPM, '
               '4–6 mice per group). Programs are also built with 91\'s external-program rule (p ≤ 0.01 and '
               '|log2FC| ≥ 0.5, unadjusted), whose noise share is the table\'s chance count over its p ≤ 0.01 count.')
RULES = {'frozen (q ≤ 0.05)': X_PROG, 'post hoc (p ≤ 0.01)': X_PROG_PH}
pd.concat({f'{k[0]} | {k[1]}': v for k, v in X_DE.items()}).to_csv(tables / 'x_lcm_de.csv.gz')

# X1: where each program sits on the healthy axis (90's set positions: centre of mass against random sets)
U_IDX = np.flatnonzero(m['usable'])
U_GENES = np.array(m['genes'])[U_IDX]
COM = {n: ca.centre_of_mass(m['deviation'][n][U_IDX], CENTRES) for n in MOUSE}
RATE_M = {n: np.asarray(m['counts'][m['specimen'] == n].sum(axis=0)).ravel() / m['library'][m['specimen'] == n].sum()
          for n in MOUSE}
LOGCP_M = np.log2(np.mean([RATE_M[n] * 1e4 for n in MOUSE], axis=0) + 1e-6)[U_IDX]
PERMS_U = pp.strata_permutations(pd.qcut(LOGCP_M, 10, labels=False, duplicates='drop'), N_PERM, np.random.default_rng(SEED))
upos = pd.Series(np.arange(len(U_GENES)), index=U_GENES)
FP_U = np.array([g not in BUILT['hvm'] for g in U_GENES])
X1 = []
for rule, progs in RULES.items():
    for k, genes in progs.items():
        mem = np.sort(upos.reindex(genes).dropna().astype(int).to_numpy())
        for label_, mm in (('all', mem), ('fold-protected', mem[FP_U[mem]])):
            if len(mm) < MIN_SET:
                X1.append({'rule': rule, 'program': k, 'members': label_, 'n': len(mm), 'p': np.nan})
                continue
            row = {'rule': rule, 'program': k, 'members': label_, 'n': len(mm)}
            for n in MOUSE:
                row[f'CoM {n}'], row[f'z {n}'], row[f'p {n}'] = pdc.set_mean_z(COM[n], mm, PERMS_U)
            same = np.sign(row[f'z {MOUSE[0]}']) == np.sign(row[f'z {MOUSE[1]}'])
            row['p'] = max(row[f'p {n}'] for n in MOUSE) if same else 1.
            X1.append(row)
X1 = pd.DataFrame(X1)
X1['q'] = np.nan
for rule in RULES:
    for label_ in ('all', 'fold-protected'):
        ok = X1['rule'].eq(rule) & X1['members'].eq(label_) & X1['p'].notna()
        if ok.any():
            X1.loc[ok, 'q'] = multipletests(X1.loc[ok, 'p'], method='fdr_bh')[1]
for n in MOUSE:
    if f'z {n}' not in X1:
        X1[f'z {n}'] = np.nan
X1['direction'] = np.where(X1[[f'z {n}' for n in MOUSE]].mean(axis=1) > 0, 'late', 'early')
X1['positioned'] = X1['q'] <= Q
display(X1.round(3))
X1.to_csv(tables / 'x1_program_positions.csv', index=False)

# X2: the programs in the injury decomposition (91's level and location rules, BH within these programs)
def a_delta(case, ref):
    d = np.mean([A_RATES[n] for n in case], axis=0) - np.mean([A_RATES[n] for n in ref], axis=0)
    return pp.centre_by_median(d, A_REF)


A_CONTRASTS = {'condition': (IRS, CONTROLS), 'R1': (['IR2A2', 'Ctrl1A2'], ['IR2A4', 'Ctrl1A4']),
               'R2': (['IR2A2', 'Ctrl1A4'], ['IR2A4', 'Ctrl1A2']),
               'IR2A2 vs controls': (['IR2A2'], CONTROLS), 'IR2A4 vs controls': (['IR2A4'], CONTROLS)}
A_PAIRS = [(i, c) for i in IRS for c in CONTROLS]
for i, c in A_PAIRS:
    A_CONTRASTS[f'{i}-{c}'] = ([i], [c])
A_DEC = {k: pdc.level_location(a_delta(*v)) for k, v in A_CONTRASTS.items()}
assert np.allclose(A_DEC['condition'][0], I91['L'].to_numpy(), atol=1e-9)
PERMS_A = pp.strata_permutations(STRATA, N_PERM, np.random.default_rng(SEED + 1))
aidx = pd.Series(np.arange(len(A_GENES)), index=A_GENES)


def rms(v):
    return float(np.sqrt(np.nanmean(v ** 2)))


def x2_row(name, mm):
    L_, P_, M_ = A_DEC['condition']
    out = {'program': name, 'members': len(mm)}
    out['L'], out['z L'], out['p L'] = pdc.set_mean_z(L_, mm, PERMS_A)
    for i, c in A_PAIRS:
        out[f'L {i}-{c}'] = float(A_DEC[f'{i}-{c}'][0][mm].mean())
    out['L R1'], out['L R2'] = float(A_DEC['R1'][0][mm].mean()), float(A_DEC['R2'][0][mm].mean())
    prof, out['M'], out['z M'], out['p M'] = pdc.set_profile_z(P_, mm, PERMS_A)
    out['peak'] = float(CENTRES[np.nanargmax(np.abs(prof))])
    out['peak sign'] = float(np.sign(prof[np.nanargmax(np.abs(prof))]))
    a, b = (np.nanmean(A_DEC[k][1][mm], axis=0) for k in ('IR2A2 vs controls', 'IR2A4 vs controls'))
    ok = np.isfinite(a) & np.isfinite(b)
    out['r IR mice'] = float(np.corrcoef(a[ok], b[ok])[0, 1])
    out['M R1'], out['M R2'] = rms(np.nanmean(A_DEC['R1'][1][mm], axis=0)), rms(np.nanmean(A_DEC['R2'][1][mm], axis=0))
    return out, prof


X2_ROWS, X2_PROF = [], {}
for rule, progs in RULES.items():
    for k, genes in progs.items():
        mm = np.sort(aidx.reindex(genes).dropna().astype(int).to_numpy())
        if len(mm) < MIN_SET:
            X2_ROWS.append({'rule': rule, 'program': k, 'members': len(mm), 'p L': np.nan, 'p M': np.nan})
            continue
        r, prof = x2_row(k, mm)
        X2_ROWS.append({'rule': rule, **r})
        X2_PROF[(rule, k)] = prof
X2 = pd.DataFrame(X2_ROWS)
for col in ['L', 'z L', 'M', 'z M', 'peak', 'r IR mice', 'L R1', 'L R2', 'M R1', 'M R2'] + [f'L {i}-{c}' for i, c in A_PAIRS]:
    if col not in X2:
        X2[col] = np.nan
for st in ('L', 'M'):
    X2[f'q {st}'] = np.nan
    for rule in RULES:
        ok = X2['rule'].eq(rule) & X2[f'p {st}'].notna()
        if ok.any():
            X2.loc[ok, f'q {st}'] = multipletests(X2.loc[ok, f'p {st}'], method='fdr_bh')[1]
ok = X2['p L'].notna()
pair_ok = np.all([np.sign(X2[f'L {i}-{c}']) == np.sign(X2.L) for i, c in A_PAIRS], axis=0)
X2['level set'] = ok & (X2['q L'] <= Q) & pair_ok & (X2['L R1'].abs() <= RELABEL * X2.L.abs()) & (X2['L R2'].abs() <= RELABEL * X2.L.abs())
X2['location set'] = ok & (X2['q M'] <= Q) & (X2['r IR mice'] >= R_REP) & (X2['M R1'] <= RELABEL * X2.M) & (X2['M R2'] <= RELABEL * X2.M)
display(X2.round(3))
X2.to_csv(tables / 'x2_programs_in_injury.csv', index=False)

# X3: gene-level agreement of the dapagliflozin S3 change (non-diabetic) with the IR change at the late PT
LATE_MASK = A_VALID & (CENTRES >= 1.95)
late = {k: np.nanmean(a_delta(*A_CONTRASTS[k])[:, LATE_MASK], axis=1) for k in ('condition', 'R1', 'R2')}
X3 = []
for (seg, cname), de in X_DE.items():
    g = de.index.intersection(A_GENES)
    row = {'segment': seg, 'contrast': cname, 'genes': len(g)}
    for k in ('condition', 'R1', 'R2'):
        row[f'rho with Δ late ({k})'] = spearmanr(de.loc[g, 'lfc'], late[k][aidx[g].to_numpy()]).statistic
        row[f'rho with L ({k})'] = spearmanr(de.loc[g, 'lfc'], A_DEC[k][0][aidx[g].to_numpy()]).statistic
    X3.append(row)
X3 = pd.DataFrame(X3)
display(X3.round(3))
X3.to_csv(tables / 'x3_gene_agreement.csv', index=False)

## 9b · M · How glucose moves along the PT: a balance driven by the measured profiles (illustration)

Relative SGLT2 and SGLT1 densities per bin are 2^(mean control deviation), scaled to a maximum of 1; MAP17 stands in
for SGLT2 as the fold-protected version. The step version replaces each profile by its mean within S1, S2 and S3.
Each version is calibrated separately: with SGLT2 absent the PT reabsorbs 40% of the load at 8 mM (mouse: 36% in
Sglt2−/−, 44% at high empagliflozin), and 1% escapes at 16 mM. Outputs: the luminal concentration along the PT, the
share taken up by SGLT1 and in S3, normally and with SGLT2 90% inhibited, at 8 and 20 mM.

In [ ]:
def density(g):
    d = np.nanmean([m['deviation'][n][m['index'][g]] for n in MOUSE], axis=0)
    d = pd.Series(d).interpolate(limit_direction='both').to_numpy()
    return np.exp2(d - d.max())


SEG_OF_BIN = np.digitize(CENTRES, [1., 2.])


def stepped(s):
    return np.array([s[SEG_OF_BIN == k].mean() for k in range(3)])[SEG_OF_BIN]


DENS = {'SGLT2': density('Slc5a2'), 'MAP17': density('Pdzk1ip1'), 'SGLT1': density('Slc5a1')}
VERSIONS = {}
for name in ('SGLT2', 'MAP17'):
    VERSIONS[f'continuous ({name})'] = (DENS[name], DENS['SGLT1'])
    VERSIONS[f'step ({name})'] = (stepped(DENS[name]), stepped(DENS['SGLT1']))
VERSIONS['literature step'] = ((SEG_OF_BIN < 2).astype(float), (SEG_OF_BIN == 2).astype(float))
READING.append('M post hoc: a "literature step" version with SGLT2 only in S1–S2 and SGLT1 only in S3, the layout of the '
               'published nephron models (Layton 2016, Hu 2021), calibrated the same way. The frozen step version keeps '
               'the measured profiles\' segment means, so it cannot show what the published layout assumes.')
VARIANTS = {'base': {}, 'no water reabsorption': {'flow_end': 1.}, 'K0.5 halved': {'k2': 2.45, 'k1': .9},
            'densities below 5% set to 0': {'floor': .05}, 'threshold 12 mM': {'threshold': 12.}}


def model_rows():
    rows, curves = [], {}
    for vname, opts in VARIANTS.items():
        kw = {k: v for k, v in opts.items() if k in ('flow_end', 'k2', 'k1')}
        for name, (s2, s1) in VERSIONS.items():
            if 'floor' in opts:
                s2, s1 = np.where(s2 < opts['floor'], 0., s2), np.where(s1 < opts['floor'], 0., s1)
            a2, a1 = gh.calibrate_capacities(s2, s1, block_reabsorbed=.4, threshold=opts.get('threshold', 16.), **kw)
            for c0 in (8., 20.):
                for inhib in (0., .9, 1.):
                    F, u2, u1 = gh.luminal_glucose(s2, s1, a2 * (1 - inhib), a1, c0=c0, **kw)
                    q = 1. - (1. - kw.get('flow_end', 1 / 3)) * np.arange(NB + 1) / NB
                    conc = c0 * F / q
                    edge = {x: float(np.interp(x, np.r_[0., CENTRES + .05], 1 - F)) for x in (.5, 1., 2.)}
                    rows.append({'variant': vname, 'version': name, 'c0 mM': c0, 'SGLT2 inhibition': inhib, 'a2': a2, 'a1': a1,
                                 'reabsorbed': 1 - F[-1], 'by SGLT1': u1.sum(), 'in S3': (u1 + u2)[SEG_OF_BIN == 2].sum(),
                                 'reabsorbed by 0.5': edge[.5], 'reabsorbed by S1/S2': edge[1.], 'reabsorbed by S2/S3': edge[2.],
                                 'luminal mM at S2/S3': float(np.interp(2., np.r_[0., CENTRES + .05], conc)),
                                 'share of reabsorption before S2/S3': edge[2.] / (1 - F[-1])})
                    if vname == 'base':
                        curves[(name, c0, inhib)] = conc
    return pd.DataFrame(rows), curves


MODEL, MODEL_CURVES = model_rows()
base = MODEL[MODEL.variant.eq('base')]
display(base.round(3))
MODEL.to_csv(tables / 'm_glucose_balance.csv', index=False)
M_CMP = MODEL.pivot_table(index=['variant', 'c0 mM', 'SGLT2 inhibition'], columns='version',
                          values=['by SGLT1', 'in S3', 'luminal mM at S2/S3'])
display(M_CMP.round(3))
# M-check (protocol): with SGLT2 absent, Sglt2−/− micropuncture puts 21% of 36% (0.58) of reabsorption before the late
# proximal tubule (Vallon 2011, Figs 5–6)
M_CHECK = MODEL[MODEL['SGLT2 inhibition'].eq(1.) & MODEL['c0 mM'].eq(8.)].pivot_table(
    index='variant', columns='version', values='share of reabsorption before S2/S3')
display(M_CHECK.round(3))
M_CHECK.to_csv(tables / 'm_check_sglt2_absent.csv')

## 10 · Predictions and decision rules (protocol section 4)

In [ ]:
E90 = D90.reindex(list(GENES))


def mouse_e(g):
    """90's per-mouse end contrast (8 bins at each end of the deviation profile) for any expressed mouse gene."""
    return np.array([ca.end_contrast(m['deviation'][n][m['index'][g]], 8)[0] for n in MOUSE])


assert np.allclose(mouse_e('Slc5a2'), E90.loc['Slc5a2', ['E Ctrl1A2', 'E Ctrl1A4']].to_numpy(float)), 'G9: E differs from 90'
READING.append('HP post hoc: 90\'s end contrast exists only for genes measured in both species, so mouse-only genes '
               '(SGLT1, G6pc) get the same per-mouse end contrast computed here (guard: it reproduces 90\'s E for SGLT2).')


def mouse_dir(g):
    if g in E90.index and np.isfinite(E90.at[g, 'E Ctrl1A2']):
        e = E90.loc[g, ['E Ctrl1A2', 'E Ctrl1A4']].to_numpy(float)
    elif g in m['index'].index:
        e = mouse_e(g)
    else:
        return 'not testable'
    return 'falls' if (e <= -.25).all() else 'rises' if (e >= .25).all() else 'mixed or flat'


def human_dir(g):
    if g not in E90.index or not DETECT.at[g, 'human usable']:
        return 'not testable'
    e = E90.at[g, 'E human']
    return 'falls' if e <= -.25 else 'rises' if e >= .25 else 'flat'


def observed(g):
    r = I_G.loc[g]
    return f"L {r['L']:.2f}, M {r['M']:.2f}, call {r['call']}, peak {r['peak']:.2f}, Δ late {r['Δ late']:.2f}"


def level_verdict(g, sign):
    if g not in I_G.index or not np.isfinite(I_G.at[g, 'L']):
        return {'verdict': 'not testable'}
    r = I_G.loc[g]
    called = r['call'] in ('level', 'both')
    same = np.sign(r['L']) == sign
    return {'observed': observed(g),
            'verdict': 'holds' if called and same else 'fails' if called else 'direction only' if same else 'opposite, no call'}


def location_verdict(g, sign, lo, hi=9.):
    if g not in I_G.index or not np.isfinite(I_G.at[g, 'L']):
        return {'verdict': 'not testable'}
    r = I_G.loc[g]
    j = A_INDEX[g]
    late = np.nanmean(DELTA[j, A_VALID & (CENTRES >= lo) & (CENTRES < hi)])
    called = r['call'] in ('location', 'both')
    in_place = lo <= r['peak'] < hi
    same = np.sign(late) == sign
    return {'observed': observed(g) + f', Δ in stretch {late:.2f}',
            'verdict': ('holds' if called and in_place and same else 'call elsewhere' if called and same else
                        'fails' if called else 'direction only' if same else 'opposite, no call')}


P_ROWS = []
for g in ('Slc5a2', 'Pdzk1ip1', 'Slc2a2'):
    P_ROWS.append({'id': 'HP1', 'gene': g, 'expect': 'falls', 'observed': mouse_dir(g)})
for g in ('Slc5a1', 'Slc2a1'):
    P_ROWS.append({'id': 'HP2', 'gene': g, 'expect': 'rises', 'observed': mouse_dir(g)})
for g in [g for g in HANDOFF['down'] if g in H2_GENES]:
    t = H2[H2.gene.eq(g)]
    P_ROWS.append({'id': 'HP3', 'gene': g, 'expect': 'midpoint < 1.85 in both mice and both exports',
                   'observed': f"{t['S position'].min():.2f}–{t['S position'].max():.2f}",
                   'verdict': 'holds' if (t['S position'] < 1.85).all() else 'fails'})
for e, names in (('human vs mouse', MOUSE), ('AKI', CONTROLS)):
    t = H2[H2.export.eq(e)]
    d = [abs(t[(t.mouse == n) & (t.gene == 'Pdzk1ip1')]['S position'].iloc[0] - t[(t.mouse == n) & (t.gene == 'Slc5a2')]['S position'].iloc[0])
         for n in names]
    P_ROWS.append({'id': 'HP4', 'gene': f'Pdzk1ip1 vs Slc5a2 ({e})', 'expect': '|Δ midpoint| ≤ 0.3 in both mice',
                   'observed': ', '.join(f'{v:.2f}' for v in d), 'verdict': 'holds' if max(d) <= .3 else 'fails'})
for g in ('Pck1', 'G6pc', 'Fbp1', 'Pcx', 'Aldob'):
    P_ROWS.append({'id': 'HP5', 'gene': g, 'expect': 'falls', 'observed': mouse_dir(g)})
for g in ('Hk1', 'Hk2'):
    P_ROWS.append({'id': 'HP5', 'gene': g, 'expect': 'rises', 'observed': mouse_dir(g)})
for g in ('Slc5a2', 'Pdzk1ip1', 'Slc2a2'):
    P_ROWS.append({'id': 'SP1', 'gene': g, 'expect': 'falls', 'observed': human_dir(g)})
for g in ('Slc5a2', 'Pdzk1ip1', 'Slc2a2'):
    obs = (f"human {S.at[g, 'centre of mass (human, equal bins)']:.3f} vs mouse {S.at[g, 'centre of mass (mouse, equal bins)']:.3f}"
           if g in S.index and np.isfinite(S.loc[g].get('centre of mass (human, equal bins)', np.nan))
           and np.isfinite(S.loc[g].get('centre of mass (mouse, equal bins)', np.nan)) else 'not testable')
    P_ROWS.append({'id': 'SP2', 'gene': g, 'expect': 'descriptive', 'observed': obs, 'verdict': 'reported'})
P_ROWS.append({'id': 'IP1', 'gene': 'Slc5a2', 'expect': 'level down', **level_verdict('Slc5a2', -1)})
P_ROWS.append({'id': 'IP2', 'gene': 'Slc5a1', 'expect': 'falls, location peak in S3',
               **location_verdict('Slc5a1', -1, 1.95)})
for g in ('Pdzk1ip1', 'Slc2a2'):
    ok = g in A_INDEX.index and 'Slc5a2' in A_INDEX.index
    r = float(row_corr(DELTA[[A_INDEX[g]]], DELTA[[A_INDEX['Slc5a2']]])[0]) if ok else np.nan
    same = ok and np.sign(I_G.at[g, 'L']) == np.sign(I_G.at['Slc5a2', 'L'])
    P_ROWS.append({'id': 'IP3', 'gene': g, 'expect': 'same sign of L as Slc5a2 and Δ(x) r ≥ 0.5',
                   'observed': f'r {r:.2f}, L {I_G.at[g, "L"]:.2f}' if ok else 'not testable',
                   'verdict': 'holds' if same and r >= .5 else 'fails' if ok else 'not testable'})
P_ROWS.append({'id': 'IP4', 'gene': 'Slc2a1', 'expect': 'rises, peak in S3', **location_verdict('Slc2a1', 1, 1.95)})
for g in ('Pck1', 'G6pc', 'Fbp1', 'Pcx', 'Aldob'):
    P_ROWS.append({'id': 'IP5', 'gene': g, 'expect': 'level down', **level_verdict(g, -1)})
    if g in I_G.index and bool(I_G.at[g, 'zonated']):
        a = I_G.at[g, 'A condition']
        P_ROWS.append({'id': 'IP5', 'gene': f'{g} gradient', 'expect': 'A < 0.6', 'observed': f'{a:.2f}',
                       'verdict': 'holds' if a < .6 else 'fails'})
for g in ('Hk1', 'Hk2', 'Pfkp', 'Pfkfb3', 'Pkm', 'Ldha'):
    P_ROWS.append({'id': 'IP6', 'gene': g, 'expect': 'rises, peak ≥ 1.75', **location_verdict(g, 1, 1.75)})
P_ROWS.append({'id': 'IP7', 'gene': 'Slc5a12', 'expect': 'level down', **level_verdict('Slc5a12', -1)})
for g in ('Slc16a1', 'Slc16a3'):
    P_ROWS.append({'id': 'IP7', 'gene': g, 'expect': 'rises, peak in S3', **location_verdict(g, 1, 1.95)})
if 'Hmgcs2' in I_G.index and np.isfinite(I_G.at['Hmgcs2', 'L']):
    r = I_G.loc['Hmgcs2']
    up = level_verdict('Hmgcs2', 1)['verdict']
    early = (r['call'] in ('location', 'both') and r['peak'] < 1.95) or r['Δ late'] < r['L']
    P_ROWS.append({'id': 'IP8', 'gene': 'Hmgcs2', 'expect': 'rises, more in S1/S2 than S3',
                   'observed': f"L {r['L']:.2f}, Δ late {r['Δ late']:.2f}, peak {r['peak']:.2f}",
                   'verdict': f"{up}; {'S1/S2-weighted' if early else 'not S1/S2-weighted'}"})
else:
    P_ROWS.append({'id': 'IP8', 'gene': 'Hmgcs2', 'expect': 'rises', 'verdict': 'not testable'})
for s_, sign, loc in (('Reactome_2022::Gluconeogenesis R-HSA-70263', -1, None),
                      ('Reactome_2022::Cellular Hexose Transport R-HSA-189200', -1, None),
                      ('Reactome_2022::Glycolysis R-HSA-70171', 1, True), ('MSigDB_Hallmark_2020::Glycolysis', 1, True),
                      ('KEGG_2019_Mouse::HIF-1 signaling pathway', 1, True), ('MSigDB_Hallmark_2020::Hypoxia', 1, True)):
    if s_ not in set(LIB91.set):
        P_ROWS.append({'id': 'IP9', 'gene': short(s_), 'expect': '', 'verdict': 'not testable'})
        continue
    r = LIB91.set_index('set').loc[s_]
    lvl = bool(r['level set']) and np.sign(r['L']) == sign
    obs = f"L {r['L']:.2f} (z {r['z L']:.1f}), M z {r['z M']:.1f}, peak {r['peak']:.2f}"
    if loc:
        ok = bool(r['location set']) and r['peak'] >= 1.95
        verdict = ('holds' if lvl and ok else 'level only' if lvl else 'location only' if ok else
                   'direction only' if np.sign(r['L']) == sign else 'opposite')
        expect = 'level up, location peak in S3'
    else:
        verdict = 'holds' if lvl else 'direction only' if np.sign(r['L']) == sign else 'opposite'
        expect = 'level down'
    P_ROWS.append({'id': 'IP9', 'gene': short(s_), 'expect': expect, 'observed': obs, 'verdict': verdict})
fp0 = X0.set_index('genes').loc['fold-protected']
P_ROWS.append({'id': 'XP0', 'gene': 'LCM S3 − S1 vs end contrast E (fold-protected)', 'expect': 'rho > 0, 95% interval above 0',
               'observed': f"rho {fp0['rho with end contrast E']:.2f} ({fp0['2.5%']:.2f}–{fp0['97.5%']:.2f}), n {int(fp0['n'])}",
               'verdict': 'holds' if fp0['2.5%'] > 0 else 'fails'})
xk = 'dapagliflozin (non-diabetic) S3: up'
for rule in RULES:
    xid = 'XP2' if rule.startswith('frozen') else 'XP2 (post hoc rule)'
    sel = X2[X2.rule.eq(rule) & X2.program.eq(xk)]
    xr = sel.iloc[0] if len(sel) else None
    if xr is None or not np.isfinite(xr.get('L', np.nan)):
        P_ROWS.append({'id': xid, 'gene': xk, 'expect': 'level up and location peak in S3',
                       'observed': f"members {int(xr['members']) if xr is not None else 0}", 'verdict': 'not testable'})
    else:
        lvl = bool(xr['level set']) and xr['L'] > 0
        loc = bool(xr['location set']) and xr['peak'] >= 1.95
        P_ROWS.append({'id': xid, 'gene': xk, 'expect': 'level up and location peak in S3',
                       'observed': f"L {xr['L']:.2f} (z {xr['z L']:.1f}), M z {xr['z M']:.1f}, peak {xr['peak']:.2f}, members {int(xr['members'])}",
                       'verdict': 'holds' if lvl and loc else 'level only' if lvl else 'location only' if loc else
                       'direction only' if xr['L'] > 0 else 'opposite'})
x3 = X3[X3.segment.eq('S3') & X3.contrast.eq('dapagliflozin (non-diabetic)')].iloc[0]
c, r1, r2 = (x3[f'rho with Δ late ({k})'] for k in ('condition', 'R1', 'R2'))
P_ROWS.append({'id': 'XP3', 'gene': 'dapagliflozin S3 change vs IR Δ late', 'expect': 'rho > 0 and above both relabelings',
               'observed': f'rho {c:.3f}; relabelings {r1:.3f}, {r2:.3f}; genes {int(x3.genes)}',
               'verdict': 'holds' if c > 0 and c > max(r1, r2) else 'fails'})
PRED = pd.DataFrame(P_ROWS)
direction = PRED.verdict.isna()
PRED.loc[direction, 'verdict'] = np.where(PRED.loc[direction, 'observed'].eq(PRED.loc[direction, 'expect']), 'holds',
                                          np.where(PRED.loc[direction, 'observed'].eq('not testable'), 'not testable', 'fails'))
display(PRED)
PRED.to_csv(tables / 'predictions.csv', index=False)

## 11 · Figure 92a · The healthy glucose map, mouse and human

In [ ]:
ORDER = sorted([g for g in GENES if DETECT.at[g, 'mouse usable']],
               key=lambda g: (ROLE_ORDER.index(GENES[g][1]), H1.at[g, 'centre of mass'] if g in H1.index else 9))
MEAN_DEV = np.vstack([np.nanmean([m['deviation'][n][m['index'][g]] for n in MOUSE], axis=0) for g in ORDER])
H_ORDER = [g for g in ORDER if DETECT.at[g, 'human usable']]
HUM_DEV = np.vstack([np.nanmean([hh['deviation'][n][hh['index'][HUMAN_SYMBOL[g]]] for n in HUMAN], axis=0) for g in H_ORDER])
LABEL_E = ['Slc5a2', 'Pdzk1ip1', 'Slc2a2', 'Slc5a12', 'Slc16a1', 'Bdh1', 'Txnip', 'Pck1', 'Slc5a10', 'Oxct1']


def heat(ax, cax, values, rows, names, extent_x, xlabel, cbar_label):
    lim = np.nanpercentile(np.abs(values), 98)
    im = ax.imshow(values, aspect='auto', cmap='RdBu_r', norm=TwoSlopeNorm(0, -lim, lim),
                   extent=(0, extent_x, len(rows) - .5, -.5), interpolation='nearest')
    ax.set_yticks(range(len(rows)), names, fontsize=5)
    for t_, g in zip(ax.get_yticklabels(), rows):
        t_.set_color(ROLE_COLOR[GENES[g][1]])
    ax.set_xlabel(xlabel)
    fig.colorbar(im, cax=cax, label=cbar_label)


fig = plt.figure(figsize=(183 * MM, 200 * MM))
outer = fig.add_gridspec(3, 1, height_ratios=[1, 2.1, 1.25], hspace=.42)
top = outer[0].subgridspec(1, 6, wspace=.55)
for k, g in enumerate(H2_GENES):
    ax = fig.add_subplot(top[0, k])
    for n in MOUSE:
        ax.plot(CENTRES, m['deviation'][n][m['index'][g]], color=SPEC_COLOR[n], lw=.9)
    for _, r in H2[H2.export.eq('human vs mouse') & H2.gene.eq(g)].iterrows():
        ax.axvline(r['S position'], color=SPEC_COLOR[r.mouse], lw=.5, ls=':')
    seg_lines(ax)
    ax.set_title(GENES[g][0])
    ax.set_xlim(0, 2.5)
    if k == 0:
        ax.set_ylabel('log2 deviation')
        ax.set_xlabel('PT position (segment units)', loc='left')
        panel(ax, 'a')
ax = fig.add_subplot(top[0, 5])
yk = 0
for e, names, mk in (('human vs mouse', MOUSE, 'o'), ('AKI', CONTROLS, 's')):
    for n in names:
        t = H2[H2.export.eq(e) & H2.mouse.eq(n)].set_index('gene').reindex(H2_GENES)
        ys = np.arange(len(H2_GENES)) + yk * .18 - .27
        ax.errorbar(t['S position'], ys, xerr=[t['S position'] - t['position 2.5%'], t['position 97.5%'] - t['S position']],
                    fmt=mk, ms=2.5, color=SPEC_COLOR[n], elinewidth=.6, capsize=0, mfc='white' if e == 'AKI' else SPEC_COLOR[n])
        yk += 1
seg_lines(ax)
ax.set_yticks(range(len(H2_GENES)), [GENES[g][0] for g in H2_GENES])
ax.yaxis.tick_right()
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(True)
ax.invert_yaxis()
ax.set_xlim(0, 2.5)
ax.set_xlabel('switch midpoint')
panel(ax, 'b')

mid = outer[1].subgridspec(1, 5, width_ratios=[10, .3, 2.6, 10, .3], wspace=.08)
ax, cax = fig.add_subplot(mid[0, 0]), fig.add_subplot(mid[0, 1])
heat(ax, cax, MEAN_DEV, ORDER, [label(g) for g in ORDER], 2.5, 'mouse PT position (segment units)', 'log2 deviation')
seg_lines(ax)
panel(ax, 'c')
ax, cax = fig.add_subplot(mid[0, 3]), fig.add_subplot(mid[0, 4])
heat(ax, cax, HUM_DEV, H_ORDER, [HUMAN_SYMBOL[g] for g in H_ORDER], 1., 'human PT position (fraction of the axis)',
     'log2 deviation')
panel(ax, 'd')

bottom = outer[2].subgridspec(1, 3, wspace=.75)
ax = fig.add_subplot(bottom[0, 0])
s_ok = S.dropna(subset=['E mouse', 'E human'])
ax.axhline(0, color=LIGHT, lw=.6)
ax.axvline(0, color=LIGHT, lw=.6)
ax.scatter(s_ok['E mouse'], s_ok['E human'], s=10, c=[ROLE_COLOR[GENES[g][1]] for g in s_ok.index], lw=0)
for g, r in s_ok.loc[[g for g in LABEL_E if g in s_ok.index]].iterrows():
    ax.annotate(GENES[g][0], (r['E mouse'], r['E human']), fontsize=5, xytext=(2, 1), textcoords='offset points')
ax.set_xlabel('late − early, mouse (log2)')
ax.set_ylabel('late − early, human (log2)')
panel(ax, 'e')
ax = fig.add_subplot(bottom[0, 1])
fp = pd.read_csv(T90 / 'b_set_positions_fold_protected.csv')
ax.scatter(fp['z mean'], -np.log10(fp['q'].clip(lower=1e-12)), s=3, color=LIGHT, lw=0)
hit = fp.set_index('set').reindex([s for s in SETS if s in set(fp.set)])
ax.scatter(hit['z mean'], -np.log10(hit['q'].clip(lower=1e-12)), s=12, color=ROLE_COLOR['production'], lw=0)
for s_, r in hit[hit['q'] <= Q].iterrows():
    ax.annotate(short(s_, 22), (r['z mean'], -np.log10(max(r['q'], 1e-12))), fontsize=5, xytext=(2, -6),
                textcoords='offset points', ha='center')
ax.axhline(-np.log10(Q), color=GREY, lw=.5, ls=':')
ax.set_xlabel('set position z (early < 0 < late)')
ax.set_ylabel('−log10 q')
panel(ax, 'f')
ax = fig.add_subplot(bottom[0, 2])
roles = [r for r in ROLE_ORDER if any(GENES[g][1] == r for g in ORDER)]
com_m = [H1.at[g, 'centre of mass'] for g in ORDER]
ax.scatter(com_m, [roles.index(GENES[g][1]) for g in ORDER], s=12, c=[ROLE_COLOR[GENES[g][1]] for g in ORDER], lw=0)
seg_lines(ax)
ax.set_yticks(range(len(roles)), roles)
ax.invert_yaxis()
ax.set_xlim(0, 2.5)
ax.set_xlabel('centre of mass (mouse)')
panel(ax, 'g')
save(fig, 'fig92a_healthy_glucose_map')

## 12 · Figure 92b · What injury does to glucose handling

In [ ]:
I_ORDER = [g for g in ORDER if g in A_INDEX.index]
D_ROWS = DELTA[[A_INDEX[g] for g in I_ORDER]]
lv = I_G.reindex(I_ORDER)
y = np.arange(len(I_ORDER))
fig = plt.figure(figsize=(183 * MM, 150 * MM))
outer = fig.add_gridspec(2, 1, height_ratios=[1.6, 1], hspace=.35)
top = outer[0].subgridspec(1, 5, width_ratios=[10, .3, 1.2, 4, 4], wspace=.12)
ax, cax = fig.add_subplot(top[0, 0]), fig.add_subplot(top[0, 1])
heat(ax, cax, D_ROWS, I_ORDER, [label(g) for g in I_ORDER], 2.5, 'PT position (segment units)', 'IR − control (log2)')
seg_lines(ax)
panel(ax, 'a')
for k, (col, rel, xl, letter) in enumerate((('L', ('L R1', 'L R2'), 'level L (log2)', 'b'),
                                            ('M', ('M R1', 'M R2'), 'location size M (log2)', 'c'))):
    ax = fig.add_subplot(top[0, 3 + k])
    ax.barh(y, lv[col], color=[ROLE_COLOR[GENES[g][1]] for g in I_ORDER], height=.7)
    for c_ in rel:
        ax.scatter(lv[c_], y, marker='|', s=14, color=INK, lw=.7)
    ax.axvline(0, color=INK, lw=.5)
    ax.set_ylim(len(I_ORDER) - .5, -.5)
    ax.set_yticks(y, [])
    ax.set_xlabel(xl)
    panel(ax, letter)
bottom = outer[1].subgridspec(1, 2, wspace=.3)
ax = fig.add_subplot(bottom[0, 0])
ax.scatter(LIB91['z L'], LIB91['z M'], s=3, color=LIGHT, lw=0)
hit = LIB91.set_index('set').reindex(SETS)
called = hit['level set'].astype(bool) | hit['location set'].astype(bool)
ax.scatter(hit['z L'], hit['z M'], s=12, color=np.where(called, ROLE_COLOR['production'], GREY), lw=0)
for s_, r in hit[called].iterrows():
    ax.annotate(short(s_, 24), (r['z L'], r['z M']), fontsize=5, xytext=(3, 1), textcoords='offset points')
ax.axvline(0, color=LIGHT, lw=.6)
ax.set_xlabel('set level z')
ax.set_ylabel('set location z')
panel(ax, 'd')
ax = fig.add_subplot(bottom[0, 1])
zon = lv[lv['zonated'].astype(bool)]
SURV_COLOR = {'reversed': '#B2182B', 'erased': '#EF8A62', 'attenuated': '#FDDBC7', 'kept': '#92C5DE',
              'amplified': '#2166AC'}
ax.bar(range(len(zon)), zon['A condition'], color=[SURV_COLOR.get(c, LIGHT) for c in zon['survival']])
for c_ in ('A R1', 'A R2'):
    ax.scatter(range(len(zon)), zon[c_], marker='_', s=30, color=INK, lw=.7)
ax.axhline(1, color=GREY, lw=.5, ls=':')
ax.axhline(0, color=INK, lw=.5)
ax.set_xticks(range(len(zon)), [GENES[g][0] for g in zon.index], rotation=60, ha='right', fontsize=5.5)
ax.set_ylabel('gradient survival A')
panel(ax, 'e')
save(fig, 'fig92b_injury_glucose_handling')

## 13 · Figure 92c · How many mice a drug experiment needs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(183 * MM, 58 * MM), gridspec_kw={'wspace': .5})
ax = axes[0]
ns = np.array(N_LIST[1:])
mid_stratum = D_STRATA.iloc[len(D_STRATA) // 2]
for _, r in D_STRATA.iterrows():
    ax.plot(ns, [gh.minimum_detectable_effect(r['SD level'], n) for n in ns], color=LIGHT, lw=.7)
for a, ls in zip(ALPHAS, ('-', '--')):
    ax.plot(ns, [gh.minimum_detectable_effect(mid_stratum['SD level'], n, alpha=a) for n in ns], color=INK, ls=ls)
ax.set_xlabel('mice per group')
ax.set_ylabel('smallest detectable level change (log2)')
ax.set_ylim(bottom=0)
panel(ax, 'a')
ax = axes[1]
ax.plot(D1['mice per group'], D1['smallest permutation p'], 'o-', color=INK, ms=3)
ax.set_yscale('log')
ax.axhline(.05, color=GREY, lw=.5, ls=':')
ax.set_xlabel('mice per group')
ax.set_ylabel('smallest permutation p')
panel(ax, 'b')
ax = axes[2]
dg = D_GENES.join(I_G[['L']])
for n, mk in zip((3, 4, 6), ('o', 's', '^')):
    ax.scatter(dg[f'MDE level n={n}'], dg['L'].abs(), s=8, marker=mk, color=[ROLE_COLOR[GENES[g][1]] for g in dg.index], lw=0)
lim3 = max(dg['L'].abs().max(), dg['MDE level n=3'].max()) * 1.05
ax.plot([0, lim3], [0, lim3], color=GREY, lw=.5, ls=':')
ax.set_xlabel('smallest detectable change (log2)')
ax.set_ylabel('|injury level change| (log2)')
ax.legend(handles=[Line2D([], [], marker=mk, ls='', color=GREY, ms=3, label=f'{n} per group')
                   for n, mk in zip((3, 4, 6), ('o', 's', '^'))], loc='lower right')
panel(ax, 'c')
save(fig, 'fig92c_drug_experiment_size')

## 14 · Figure 92d · Glucose along the PT in the balance

In [ ]:
fig = plt.figure(figsize=(183 * MM, 55 * MM))
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 1.1], wspace=.55)
ax = fig.add_subplot(gs[0, 0])
for name, col in (('SGLT2', ROLE_COLOR['uptake']), ('MAP17', '#56B4E9'), ('SGLT1', ROLE_COLOR['production'])):
    ax.plot(CENTRES, DENS[name], color=col, lw=1)
    ax.plot(CENTRES, stepped(DENS[name]), color=col, lw=.7, ls='--')
seg_lines(ax)
ax.set_xlim(0, 2.5)
ax.set_ylim(0, 1.3)
ax.set_yticks([0, .5, 1])
ax.set_xlabel('PT position (segment units)')
ax.set_ylabel('relative density')
ax.legend(handles=[Line2D([], [], color=c, label=n) for n, c in (('SGLT2', ROLE_COLOR['uptake']), ('MAP17', '#56B4E9'),
                                                                 ('SGLT1', ROLE_COLOR['production']))],
          loc='upper center', ncol=3, fontsize=5, handlelength=1.2, columnspacing=.8)
panel(ax, 'a')
xe = np.r_[0., CENTRES + .05]
for k, c0 in enumerate((8., 20.)):
    ax = fig.add_subplot(gs[0, 1 + k])
    for inhib, col in ((0., INK), (.9, '#B2182B')):
        ax.plot(xe, MODEL_CURVES[('continuous (MAP17)', c0, inhib)], color=col, lw=1)
        ax.plot(xe, MODEL_CURVES[('step (MAP17)', c0, inhib)], color=col, lw=.7, ls='--')
        ax.plot(xe, MODEL_CURVES[('literature step', c0, inhib)], color=col, lw=.7, ls=':')
    seg_lines(ax)
    ax.set_xlim(0, 2.5)
    ax.set_xlabel('PT position (segment units)')
    ax.set_ylabel(f'luminal glucose (mM), plasma {c0:.0f} mM')
    panel(ax, 'bc'[k])
ax = fig.add_subplot(gs[0, 3])
inh = MODEL[MODEL['SGLT2 inhibition'].eq(.9) & MODEL['c0 mM'].eq(8.)]
for (version, mk, mfc), dy in zip((('continuous (MAP17)', 'o', INK), ('step (MAP17)', 's', 'white'),
                                    ('literature step', 'D', GREY)), (-.15, .15, 0.)):
    v = inh[inh.version.eq(version)].set_index('variant').reindex(list(VARIANTS))
    ax.plot(v['in S3'], np.arange(len(VARIANTS)) + dy, mk, color=INK, mfc=mfc, ms=3, ls='')
ax.set_yticks(range(len(VARIANTS)), ['base', 'no water\nreabsorption', 'K0.5 halved', 'densities\n< 5% → 0', 'threshold\n12 mM'],
              fontsize=5)
ax.invert_yaxis()
ax.set_xlim(0, .45)
ax.set_xlabel('share of load taken up in S3')
panel(ax, 'd')
save(fig, 'fig92d_glucose_balance')

## 14b · Figure 92e · The drug's footprint on the axis

In [ ]:
fig = plt.figure(figsize=(183 * MM, 58 * MM))
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1.1, 1, 1], wspace=1.05)
ax = fig.add_subplot(gs[0, 0])
fpx = x0[x0['fold-protected'].astype(bool)]
ax.hexbin(fpx['E mouse'], fpx['LCM S3 − S1'], gridsize=40, cmap='Greys', mincnt=1, linewidths=0, bins='log',
          extent=(-6, 6, -4, 4))
ax.axhline(0, color=LIGHT, lw=.6)
ax.axvline(0, color=LIGHT, lw=.6)
ax.set_xlim(-6, 6)
ax.set_ylim(-4, 4)
ax.set_xlabel('late − early on the axis (log2)')
ax.set_ylabel('S3 − S1, microdissected (log2)')
panel(ax, 'a')
ax = fig.add_subplot(gs[0, 1])
x1a = X1[X1['members'].eq('all') & X1['p'].notna()].reset_index(drop=True)
for n, mk in zip(MOUSE, ('o', 's')):
    ax.scatter(x1a[f'z {n}'], range(len(x1a)), marker=mk, s=10, color=SPEC_COLOR[n], lw=0)
ax.axvline(0, color=INK, lw=.5)
ax.set_yticks(range(len(x1a)), (x1a.program.str.replace('dapagliflozin', 'dapa').str.replace(' (non-diabetic)', '')
                                + np.where(x1a.rule.str.startswith('post'), ' *', '')), fontsize=5)
ax.invert_yaxis()
ax.set_xlabel('position z on the healthy axis')
panel(ax, 'b')
ax = fig.add_subplot(gs[0, 2])
x2a = X2.dropna(subset=['z L']).reset_index(drop=True)
ax.scatter(x2a['z L'], x2a['z M'], s=14, lw=0,
           color=[ROLE_COLOR['production'] if ' S3' in k else ROLE_COLOR['uptake'] for k in x2a.program])
for k, r in x2a.iterrows():
    ax.annotate(r.program.replace('dapagliflozin', 'dapa').replace(' (non-diabetic)', ''), (r['z L'], r['z M']), fontsize=4.5,
                xytext=(0, 4 if k % 2 == 0 else -9), textcoords='offset points', ha='center')
ax.axvline(0, color=LIGHT, lw=.6)
ax.set_xlabel('IR level z')
ax.set_ylabel('IR location z')
panel(ax, 'c')
ax = fig.add_subplot(gs[0, 3])
for k, col in (('diabetes S1: down', '#2166AC'), ('dapagliflozin (diabetic) S1: up', '#B2182B')):
    mem = np.sort(upos.reindex(X_PROG_PH[k]).dropna().astype(int).to_numpy())
    prof = np.nanmean([np.nanmean(m['deviation'][n][U_IDX][mem], axis=0) for n in MOUSE], axis=0)
    ax.plot(CENTRES, prof, color=col, lw=1)
seg_lines(ax)
ax.axhline(0, color=INK, lw=.5)
ax.set_xlim(0, 2.5)
ax.set_xlabel('PT position (segment units)')
ax.set_ylabel('mean healthy deviation (log2)')
panel(ax, 'd')
save(fig, 'fig92e_drug_footprint')

## 15 · Readout and run record

In [ ]:
READOUT = []
print('\n'.join('· ' + g for g in GUARDS))
for r in READING:
    print('READING:', r)
record = {'logic': NOTEBOOK_LOGIC_VERSION, 'guards': GUARDS, 'reading': READING, 'seconds': SECONDS,
          'units': {'healthy mouse': pd.Series(m['specimen']).value_counts().to_dict(),
                    'human': pd.Series(hh['specimen']).value_counts().to_dict(),
                    'AKI': pd.Series(aspec).value_counts().to_dict()},
          'cuts': {'mouse': list(MOUSE_CUTS), 'human': None if HUMAN_CUTS is None else list(HUMAN_CUTS), 'aki': list(A_CUTS)},
          'minutes': (time.perf_counter() - NOTEBOOK_START) / 60}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=float))
print(f"Done in {record['minutes']:.1f} min")